
# 05 · NBED 위치별 분석 — median · ring 가상이미지 · PCA · 구조인자/RDF NMF

한 개의 NBED(nano-beam diffraction) 4D 스캔에서 **위치(probe)별** 로:

1. dm4 불러오기
2. 전체 **median** 회절패턴
3. 여러 **링 영역**의 가상이미지 (구조 대비)
4. 전체 패턴 **PCA** → 서로 다른 ring pattern이 몇 종류인지 (scree/성분)
5. 각 패턴 **radial 분포** I(q)
6. 배경 제거 → **구조인자 φ(q)** (환원 structure factor)
7. 구조인자 **NMF k=4**
8. 각 구조인자 **FFT(sine)** → **RDF G(r)**
9. RDF **NMF k=4**

> 모든 그림은 각 셀에서 PNG로, 그래프 데이터는 CSV로 `SAVE_DIR`에 저장됩니다.
> NMF는 비음수 입력이 필요합니다. φ(q)·G(r)은 부호가 있어 음수는 0으로 클리핑되니(성분이 다소 왜곡),
> `METHOD="pca"`로 바꿔 비교할 수 있게 해두었습니다(기본은 요청대로 `"nmf"`, k=4).


## 1) dm4 불러오기 & 설정

In [ ]:

import os
import numpy as np
import matplotlib.pyplot as plt
import fourdstem as fds

DM4_PATH   = "/home/jonghoonk918/Desktop/fdstem/NBED/scan.dm4"   # ← 단일 NBED dm4 경로
USE_SYNTHETIC = not os.path.isfile(DM4_PATH)

DET_BIN    = 2            # 검출기 비닝(메모리/속도). 합성이면 1
Q_UNIT_HINT= "1/A"
Q_PER_PX   = 0.0120       # 초기 추정(캘리브레이션으로 갱신). 실데이터는 nb1 값 참고
R_TARGET   = 1.61         # median 패턴 1st peak를 이 값(Si–O)에 맞춰 q 캘리브레이션
N_JOBS     = -2           # 환원 병렬 코어수(-2=전부-1). 위치가 많으면 수 분 걸릴 수 있음
K          = 4            # NMF 성분 수 (요청: 4)
K_PCA      = 8            # PCA 성분 수(scree로 '몇 종류'인지 판단)
N_BINS     = None         # radial bin 수 (None=자동)
METHOD     = "nmf"        # 구조인자/RDF 분해 방법: "nmf"(요청) 또는 "pca"
RINGS_PX   = [(6,12),(12,18),(18,26),(26,36)]   # 가상이미지용 링(원본 검출기 px, 비닝 자동 반영)
CFG = fds.RDFConfig(composition={"Si":1,"O":2}, q_int_min=0.20, q_int_max=1.50,
                    r_min=1.10, r_max=8.0, dr=0.02, damping="lorch")
FIRST_WIN = (1.45, 1.85)

def make_nbed_cube(scan=(40,40), dp=(96,96), seed=0):
    '''합성 데모: 스캔을 3개 영역으로 나눠 서로 다른 ring pattern(반경) 배정 + 중심빔 + 노이즈.
    PCA/NMF가 서로 다른 링 패턴을 찾아내는지 확인용.'''
    rng = np.random.default_rng(seed); Sy,Sx = scan; H,W = dp
    yy,xx = np.mgrid[0:H,0:W]; cx,cy=W/2,H/2; rr=np.hypot(xx-cx,yy-cy)
    def ring(r0,a=1.0,s=3.0): return a*np.exp(-(rr-r0)**2/(2*s**2))
    beam = 6*np.exp(-rr**2/(2*2.0**2))
    phases = [ring(20)+0.4*ring(34), ring(26)+0.3*ring(40), ring(16)+0.5*ring(30)]
    cube = np.empty((Sy,Sx,H,W), np.float32)
    for iy in range(Sy):
        for ix in range(Sx):
            ph = 0 if ix < Sx//3 else (1 if ix < 2*Sx//3 else 2)
            cube[iy,ix] = beam + phases[ph]*(1+0.1*rng.standard_normal()) \
                          + 0.8*rng.standard_normal((H,W))
    return np.clip(cube,0,None)

if USE_SYNTHETIC:
    cube = fds.from_array(make_nbed_cube(), q_per_px=Q_PER_PX, name="synthetic-NBED")
else:
    cube = fds.load(DM4_PATH, Q_UNIT_HINT)
    if DET_BIN > 1: cube = fds.bin_cube_detector(cube, DET_BIN)
scan = cube.scan_shape; dp = cube.dp_shape
b = 1 if USE_SYNTHETIC else max(DET_BIN,1)
rings = [(ri/b, ro/b) for ri,ro in RINGS_PX]
print("USE_SYNTHETIC =", USE_SYNTHETIC, "| cube:", cube.shape, "| scan:", scan, "| dp:", dp)

# 저장 위치 & 헬퍼
SAVE_DIR = (os.path.dirname(DM4_PATH)+"/nb5_outputs" if not USE_SYNTHETIC else "nb5_outputs")
os.makedirs(SAVE_DIR, exist_ok=True)
def save(fig, name):
    p=os.path.join(SAVE_DIR,name+".png"); fig.savefig(p,dpi=150,bbox_inches="tight"); print("saved:",p)
def save_csv(name, header, rows):
    import csv; p=os.path.join(SAVE_DIR,name+".csv")
    with open(p,"w",newline="") as f:
        w=csv.writer(f); w.writerow(header); w.writerows(rows)
    print("saved:", p)
print("outputs ->", os.path.abspath(SAVE_DIR))



## 2) 전체 median 회절패턴

평균 대신 **median** — X선 히트·hot frame·간헐 Bragg 스팟 같은 이상치를 걸러낸 "대표" NBED 패턴.
여기서 중심을 찾아 이후 링/방위각에 씁니다.


In [ ]:

med = fds.median_pattern(cube)
(cx, cy), _ = fds.find_center(med, fds.beam_stopper_mask(med))
print("center:", (round(cx,1), round(cy,1)))

fig, ax = plt.subplots(1, 2, figsize=(10, 4.4))
im0 = ax[0].imshow(med, cmap="magma"); ax[0].plot(cx, cy, "c+", ms=10)
ax[0].set_title("median NBED pattern"); plt.colorbar(im0, ax=ax[0], fraction=0.046)
im1 = ax[1].imshow(np.log1p(med), cmap="magma"); ax[1].plot(cx, cy, "c+", ms=10)
ax[1].set_title("median (log)"); plt.colorbar(im1, ax=ax[1], fraction=0.046)
plt.tight_layout(); save(fig, "02_median_pattern"); plt.show()
np.save(os.path.join(SAVE_DIR, "02_median_pattern.npy"), med); print("saved: 02_median_pattern.npy")



## 3) 링 영역별 가상이미지

여러 반경 링(`RINGS_PX`)을 환형 검출기로 써서 각 링의 산란 세기를 스캔 상에 매핑합니다. 링마다 다른
상/구조가 다른 대비로 나타납니다.


In [ ]:

nR = len(rings)
fig, ax = plt.subplots(1, nR+1, figsize=(3.0*(nR+1), 3.2))
ax[0].imshow(np.log1p(med), cmap="magma")
for ri,ro in rings:
    th=np.linspace(0,2*np.pi,100)
    for rad in (ri,ro): ax[0].plot(cx+rad*np.cos(th), cy+rad*np.sin(th), "c-", lw=0.6)
ax[0].set_title("rings on median"); ax[0].axis("off")
vimgs=[]
for j,(ri,ro) in enumerate(rings):
    vi = np.asarray(fds.annular_dark_field(cube, center=(cx,cy), r_inner=ri, r_outer=ro))
    vimgs.append(vi)
    im=ax[j+1].imshow(vi, cmap="viridis"); ax[j+1].set_title(f"ring {int(ri)}-{int(ro)} px", fontsize=9)
    ax[j+1].axis("off")
plt.tight_layout(); save(fig, "03_ring_virtual_images"); plt.show()
# 저장: 각 링 가상이미지(스캔 평탄화 컬럼)
save_csv("03_ring_virtual_images",
         ["scan_index"]+[f"ring_{int(ri)}_{int(ro)}" for ri,ro in rings],
         [[i]+[f"{v.ravel()[i]:.6g}" for v in vimgs] for i in range(vimgs[0].size)])



## 4) 전체 패턴 PCA — 서로 다른 ring pattern이 몇 종류?

패턴별 정규화 후 PCA. **scree(설명분산)** 의 꺾임(elbow)이 구별되는 패턴 종류 수의 힌트입니다. 각 성분의
회절패턴(무엇이 변하는지)과 loading 맵(어디서 변하는지)을 함께 봅니다.


In [ ]:

pca = fds.pca_decompose(cube, n_components=K_PCA, normalize="sum")
evr = pca.explained_variance_ratio
fig = plt.figure(figsize=(14, 6))
gs = fig.add_gridspec(3, K_PCA)
axs = fig.add_subplot(gs[0, :2])
axs.plot(np.arange(1,len(evr)+1), evr, "o-"); axs.set_xlabel("component"); axs.set_ylabel("explained var ratio")
axs.set_title("PCA scree (elbow ~ #distinct patterns)")
axc = fig.add_subplot(gs[0, 2:])
axc.plot(np.arange(1,len(evr)+1), np.cumsum(evr), "s-"); axc.set_ylim(0,1.02)
axc.set_xlabel("component"); axc.set_ylabel("cumulative"); axc.set_title("cumulative explained var")
nshow = min(K_PCA, 6)
for i in range(nshow):
    a1=fig.add_subplot(gs[1,i]); a1.imshow(pca.components[i], cmap="coolwarm"); a1.axis("off")
    a1.set_title(f"PC{i+1}", fontsize=8)
    a2=fig.add_subplot(gs[2,i]); a2.imshow(pca.loadings[i], cmap="viridis"); a2.axis("off")
    a2.set_title(f"load {i+1}", fontsize=8)
plt.tight_layout(); save(fig, "04_pca_patterns"); plt.show()
save_csv("04_pca_explained_variance",
         ["component","explained_var_ratio","cumulative"],
         [[i+1, f"{evr[i]:.6g}", f"{np.cumsum(evr)[i]:.6g}"] for i in range(len(evr))])



## 5) 위치별 radial 분포 I(q)

각 NBED 패턴을 방위각 평균 → 위치별 I(q). q 축은 median 패턴을 **1st peak = R_TARGET(1.61 Å)** 에 맞춰
캘리브레이션한 값입니다.


In [ ]:

# q 캘리브레이션: median 패턴 1st peak -> R_TARGET
def _r1(qpp):
    rr = fds.pattern_to_rdf(med, qpp, CFG, center=(cx,cy),
                            center_beam_radius=max(1, int(0.2/qpp)))
    r1,_ = fds.first_peak_position(rr.r, rr.Gr, 1.3, 2.6); return r1
QPP = cube.calibration.q_per_px or Q_PER_PX
if not USE_SYNTHETIC:
    for _ in range(6):
        ri=_r1(QPP)
        if not np.isfinite(ri) or ri<=0 or abs(ri-R_TARGET)<0.003: break
        QPP *= ri/R_TARGET
    print(f"[calib] q_per_px -> {QPP:.5g} 1/A/px  (1st peak -> {_r1(QPP):.3f} A)")
else:
    print(f"[calib] synthetic: q_per_px = {QPP:.5g}")

profiles, r_px = fds.radial_profiles(cube, (cx,cy), n_bins=N_BINS, normalize=False)
q = r_px * QPP
print("profiles:", profiles.shape, "| q range:", round(q.min(),3), "..", round(q.max(),3), "1/A")

fig, ax = plt.subplots(1, 2, figsize=(12, 4.3))
ax[0].plot(q, profiles.mean(0), "k-", lw=2, label="mean")
for i in np.linspace(0, len(profiles)-1, 6).astype(int):
    ax[0].plot(q, profiles[i], lw=0.6, alpha=0.6)
ax[0].set_xlabel("q (1/Å)"); ax[0].set_ylabel("I(q)"); ax[0].set_title("radial profiles (mean + samples)"); ax[0].legend()
im=ax[1].imshow(profiles, aspect="auto", cmap="viridis",
                extent=[q.min(), q.max(), len(profiles), 0])
ax[1].set_xlabel("q (1/Å)"); ax[1].set_ylabel("scan position index"); ax[1].set_title("I(q) — all positions")
plt.colorbar(im, ax=ax[1], fraction=0.046)
plt.tight_layout(); save(fig, "05_radial_profiles"); plt.show()
save_csv("05_radial_mean", ["q_invA","I_mean"],
         [[f"{q[i]:.5f}", f"{profiles.mean(0)[i]:.6g}"] for i in range(len(q))])



## 6) 배경 제거 → 구조인자 φ(q)

각 I(q)를 환원(scale N 자동적합, 원자 산란인자로 배경 제거)하여 **구조인자 φ(q)=q(S(q)−1)** 로 만듭니다.
위치가 많으면 병렬(`N_JOBS`)로 수 분 걸릴 수 있습니다.


In [ ]:

red = fds.reduce_profiles(profiles, q, CFG, n_jobs=N_JOBS, progress=True)
qf, phi, rr, Gr, ok = red["q"], red["phi"], red["r"], red["Gr"], red["ok"]
print("reduced ok:", int(ok.sum()), "/", len(ok), "| phi:", phi.shape, "| Gr:", Gr.shape)

fig, ax = plt.subplots(1, 2, figsize=(12, 4.3))
ax[0].axhline(0, color="0.8", lw=0.8)
ax[0].plot(qf, np.nanmean(phi[ok], 0), "k-", lw=2, label="mean")
for i in np.where(ok)[0][::max(1, ok.sum()//6)][:6]:
    ax[0].plot(qf, phi[i], lw=0.6, alpha=0.6)
ax[0].set_xlabel("q (1/Å)"); ax[0].set_ylabel("φ(q)"); ax[0].set_title("structure factor φ(q)"); ax[0].legend()
im=ax[1].imshow(phi[ok], aspect="auto", cmap="coolwarm",
                extent=[qf.min(), qf.max(), int(ok.sum()), 0],
                vmin=-np.nanpercentile(np.abs(phi[ok]),98), vmax=np.nanpercentile(np.abs(phi[ok]),98))
ax[1].set_xlabel("q (1/Å)"); ax[1].set_ylabel("position (ok only)"); ax[1].set_title("φ(q) — all positions")
plt.colorbar(im, ax=ax[1], fraction=0.046)
plt.tight_layout(); save(fig, "06_structure_factor"); plt.show()
save_csv("06_structure_factor_mean", ["q_invA","phi_mean"],
         [[f"{qf[i]:.5f}", f"{np.nanmean(phi[ok],0)[i]:.6g}"] for i in range(len(qf))])



## 7) 구조인자 NMF (k=4)

구조인자 스택을 `k=4` 성분으로 분해 → **4개의 대표 구조인자**(성분)와 각 위치의 **분율(loading 맵)**.
서로 다른 링 패턴/상이 성분으로 분리됩니다. (φ는 부호가 있어 음수는 0으로 클리핑; `METHOD="pca"`로 비교 가능)


In [ ]:

def decompose_and_map(stack, ok, k, method, xaxis):
    dpc = fds.decompose_profiles(stack[ok], n_components=k, method=method, x=xaxis)
    maps = np.full((k, int(np.prod(scan))), np.nan); idx=np.where(ok)[0]
    for i in range(k): maps[i, idx] = dpc.fractions[:, i]
    return dpc, maps.reshape(k, *scan)

dp_sf, sf_maps = decompose_and_map(phi, ok, K, METHOD, qf)
fig, ax = plt.subplots(2, K, figsize=(3.2*K, 6))
for i in range(K):
    ax[0,i].axhline(0, color="0.85", lw=0.8); ax[0,i].plot(qf, dp_sf.components[i], lw=1.4)
    ax[0,i].set_title(f"SF comp {i+1}", fontsize=9); ax[0,i].set_xlabel("q (1/Å)")
    im=ax[1,i].imshow(sf_maps[i], cmap="viridis"); ax[1,i].set_title(f"fraction {i+1}", fontsize=9)
    ax[1,i].axis("off")
fig.suptitle(f"structure factor {METHOD.upper()} (k={K})", y=1.02)
plt.tight_layout(); save(fig, "07_structure_factor_nmf"); plt.show()
save_csv("07_sf_components", ["q_invA"]+[f"comp{i+1}" for i in range(K)],
         [[f"{qf[j]:.5f}"]+[f"{dp_sf.components[i][j]:.6g}" for i in range(K)] for j in range(len(qf))])



## 8) 구조인자 FFT(sine) → RDF G(r)

각 φ(q)를 사인 변환하여 위치별 **RDF G(r)** 을 얻습니다(환원 단계에서 함께 계산됨).


In [ ]:

fig, ax = plt.subplots(1, 2, figsize=(12, 4.3))
ax[0].axhline(0, color="0.8", lw=0.8)
ax[0].plot(rr, np.nanmean(Gr[ok], 0), "k-", lw=2, label="mean")
for i in np.where(ok)[0][::max(1, ok.sum()//6)][:6]:
    ax[0].plot(rr, Gr[i], lw=0.6, alpha=0.6)
ax[0].set_xlim(0, 8); ax[0].set_xlabel("r (Å)"); ax[0].set_ylabel("G(r)")
ax[0].set_title("RDF G(r)"); ax[0].legend()
im=ax[1].imshow(Gr[ok], aspect="auto", cmap="coolwarm", extent=[rr.min(), rr.max(), int(ok.sum()), 0],
                vmin=-np.nanpercentile(np.abs(Gr[ok]),98), vmax=np.nanpercentile(np.abs(Gr[ok]),98))
ax[1].set_xlim(0, 8); ax[1].set_xlabel("r (Å)"); ax[1].set_ylabel("position (ok only)")
ax[1].set_title("G(r) — all positions"); plt.colorbar(im, ax=ax[1], fraction=0.046)
plt.tight_layout(); save(fig, "08_rdf"); plt.show()
save_csv("08_rdf_mean", ["r_A","Gr_mean"],
         [[f"{rr[i]:.4f}", f"{np.nanmean(Gr[ok],0)[i]:.6g}"] for i in range(len(rr))])



## 9) RDF NMF (k=4)

RDF 스택을 `k=4` 성분으로 분해 → **4개의 대표 RDF**와 위치별 분율 맵. 구조적으로 다른 국소 배열이 성분으로
나뉩니다. (G(r)도 부호 있음 → 음수 클리핑 주의; `METHOD="pca"` 비교 가능)


In [ ]:

dp_rdf, rdf_maps = decompose_and_map(Gr, ok, K, METHOD, rr)
fig, ax = plt.subplots(2, K, figsize=(3.2*K, 6))
for i in range(K):
    ax[0,i].axhline(0, color="0.85", lw=0.8); ax[0,i].plot(rr, dp_rdf.components[i], lw=1.4)
    ax[0,i].set_xlim(0, 8); ax[0,i].set_title(f"RDF comp {i+1}", fontsize=9); ax[0,i].set_xlabel("r (Å)")
    im=ax[1,i].imshow(rdf_maps[i], cmap="viridis"); ax[1,i].set_title(f"fraction {i+1}", fontsize=9)
    ax[1,i].axis("off")
fig.suptitle(f"RDF {METHOD.upper()} (k={K})", y=1.02)
plt.tight_layout(); save(fig, "09_rdf_nmf"); plt.show()
save_csv("09_rdf_components", ["r_A"]+[f"comp{i+1}" for i in range(K)],
         [[f"{rr[j]:.4f}"]+[f"{dp_rdf.components[i][j]:.6g}" for i in range(K)] for j in range(len(rr))])
print("\nAll outputs in:", os.path.abspath(SAVE_DIR))



**정리** — 한 NBED 스캔에서 위치별로 (2) median, (3) 링 가상이미지, (4) PCA(패턴 종류), (5) radial I(q),
(6) 구조인자 φ(q), (7) φ(q) NMF k=4, (8) FFT→RDF, (9) RDF NMF k=4 를 계산·저장합니다.
- 위치 수가 많으면 (6)이 느립니다 → `N_JOBS`(코어), `DET_BIN`(비닝)으로 조절.
- φ(q)·G(r)은 부호가 있어 NMF에서 음수가 클리핑됩니다. 성분 해석이 이상하면 `METHOD="pca"`와 비교하세요.
- (4) scree의 elbow와 (7)·(9)의 성분 수(k)를 데이터에 맞춰 조정하세요.
